### Regularization and coefficient interpretation

The unregularized MNL estimates on the AI-generated booking data were unstable because the AI choices were much more deterministic than the real human choices. To stabilize the estimation, I use a ridge-regularized MNL with λ = 1. This value is not tuned for prediction or selected by cross-validation; it is used only as a stabilization device to prevent coefficient blow-up and recover the qualitative direction of the AI-implied preferences.

Because of this, I do not interpret the AI coefficient magnitudes as exact behavioral parameters. In addition, the human normalized coefficients and the AI regularized coefficients are not perfectly on the same scaling convention. Therefore, the coefficient comparison should be interpreted qualitatively: the main question is whether the signs and broad preference patterns are similar, not whether the magnitudes are directly comparable.

In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

df = pd.read_csv("1_data_with_ai_bookings_sample100_corrected.csv")
df = df[df["in_ai_sample_corrected"] == True].copy()

choice_col = "booking_ai_sample100_corrected"

features = [
    "prop_starrating",
    "prop_review_score",
    "prop_brand_bool",
    "prop_location_score",
    "prop_accesibility_score",
    "prop_log_historical_price",
    "price_usd",
    "promotion_flag"
]

df = df[["srch_id", choice_col] + features].copy()

df[features] = df[features].replace([np.inf, -np.inf], np.nan)
df[features] = df[features].fillna(df[features].median())

feature_means = df[features].mean()
feature_stds = df[features].std().replace(0, 1)

df_scaled = df.copy()
df_scaled[features] = (df_scaled[features] - feature_means) / feature_stds

groups = list(df_scaled.groupby("srch_id"))

def negative_log_likelihood(beta):
    nll = 0.0

    for srch_id, group in groups:
        X = group[features].values
        y = group[choice_col].values

        u = beta[0] + X @ beta[1:]

        max_u = max(0, np.max(u))
        exp_u = np.exp(u - max_u)
        denom = np.exp(0 - max_u) + np.sum(exp_u)

        if y.sum() == 1:
            chosen_index = np.where(y == 1)[0][0]
            log_prob = (u[chosen_index] - max_u) - np.log(denom)

        elif y.sum() == 0:
            log_prob = (0 - max_u) - np.log(denom)

        else:
            raise ValueError(f"Search {srch_id} has more than one chosen hotel.")

        nll -= log_prob

    return nll

def penalized_negative_log_likelihood(beta, lambda_penalty=1.0):
    nll = negative_log_likelihood(beta)

    # Penalize feature coefficients, but not beta_0/intercept
    ridge_penalty = lambda_penalty * np.sum(beta[1:] ** 2)

    return nll + ridge_penalty

initial_beta = np.zeros(1 + len(features))

result = minimize(
    penalized_negative_log_likelihood,
    initial_beta,
    method="BFGS"
)

beta_hat = result.x

results_table = pd.DataFrame({
    "parameter": ["beta_0"] + features,
    "estimate_ai_regularized": beta_hat
})

print(results_table)
print("Converged:", result.success)
print("Penalized negative log-likelihood:", result.fun)
print("Unpenalized negative log-likelihood:", negative_log_likelihood(beta_hat))

                   parameter  estimate_ai_regularized
0                     beta_0                -2.018489
1            prop_starrating                 1.944358
2          prop_review_score                 3.081569
3            prop_brand_bool                 0.521772
4        prop_location_score                 2.957494
5    prop_accesibility_score                 0.751137
6  prop_log_historical_price                -0.453620
7                  price_usd                -2.049738
8             promotion_flag                 0.587876
Converged: True
Penalized negative log-likelihood: 91.45884309589741
Unpenalized negative log-likelihood: 63.84623128608761


In [3]:
import pandas as pd

# Human MNL normalized coefficients from Problem 1
human_normalized = {
    "intercept": -1.9815321907864278,
    "prop_starrating": 0.4081249536151655,
    "prop_review_score": 0.10876096623704055,
    "prop_brand_bool": 0.22992269948768013,
    "prop_location_score": 0.02202632301274303,
    "prop_accesibility_score": 0.04344412341249515,
    "prop_log_historical_price": -0.06686945209512846,
    "price_usd": -1.3311099651462353,
    "promotion_flag": 0.45402977040295234
}

# AI MNL regularized standardized coefficients from Step 2
ai_normalized = {
    "intercept": -2.018489,
    "prop_starrating": 1.944358,
    "prop_review_score": 3.081569,
    "prop_brand_bool": 0.521772,
    "prop_location_score": 2.957494,
    "prop_accesibility_score": 0.751137,
    "prop_log_historical_price": -0.453620,
    "price_usd": -2.049738,
    "promotion_flag": 0.587876
}

comparison_table = pd.DataFrame({
    "Feature": list(human_normalized.keys()),
    "human value": [human_normalized[k] for k in human_normalized.keys()],
    "ai value": [ai_normalized[k] for k in human_normalized.keys()]
})

comparison_table

,Feature,human value,ai value
0,intercept,-1.981532,-2.018489
1,prop_starrating,0.408125,1.944358
2,prop_review_score,0.108761,3.081569
3,prop_brand_bool,0.229923,0.521772
4,prop_location_score,0.022026,2.957494
5,prop_accesibility_score,0.043444,0.751137
6,prop_log_historical_price,-0.066869,-0.453620
7,price_usd,-1.331110,-2.049738
8,promotion_flag,0.454030,0.587876


## Comparison

The comparison suggests that the AI-generated choices are directionally consistent with the real human choices. Both the human-data MNL and the AI-generated MNL assign positive utility to higher star ratings, better review scores, brand status, location score, accessibility score, and promotions, while assigning negative utility to higher prices and higher historical prices.

Because the AI estimates are ridge-regularized and are not perfectly on the same scaling convention as the human normalized coefficients, I do not interpret the coefficient magnitudes directly. Qualitatively, however, the AI-generated choices appear more deterministic and more strongly tied to observable hotel attributes than real human choices. This suggests that the AI agent captures broad, common-sense hotel preferences, but does not fully reproduce the noisier and more heterogeneous nature of actual customer booking behavior.